# 🧩 Interview Questions: Complex SQL Challenges
## Advanced Problem-Solving for Senior Engineers

### 🎯 Why Complex SQL Separates Senior from Staff Engineers

**Complex SQL isn't about syntax—it's about problem-solving.** Here's why mastering it matters:

1. **Proves Deep Understanding** - Anyone can write a JOIN; few can solve hierarchical queries
2. **Real Production Problems** - These patterns appear in actual data engineering work
3. **Interview Differentiator** - Complex SQL questions filter for senior+ roles
4. **Performance Impact** - Efficient solutions scale; naive ones don't
5. **Business Value** - Solving complex problems directly impacts analytics & ML

### 💡 What Separates Mid-Level from Senior Engineers

| Mid-Level Engineer | Senior Engineer |
|-------------------|------------------|
| "I'll just use multiple queries" | "One elegant recursive CTE handles this" |
| Writes procedural loops in Python | Writes set-based SQL that runs 100x faster |
| Stuck on self-referencing problems | Masters hierarchical and graph queries |
| "Can't be done in SQL" | Finds creative window function solutions |
| Focuses on getting any answer | Optimizes for performance and maintainability |

---

### 📊 Interview Question Coverage (25 Questions)

This module covers **7 advanced SQL domains**:

| Topic | Questions | Why It Matters |
|-------|-----------|----------------|
| **Recursive Queries & Hierarchies** | 4 | Org charts, bill-of-materials, graph traversal |
| **Advanced Window Functions** | 5 | Running totals, gaps/islands, streak detection |
| **Self-Joins & Complex Relationships** | 3 | Sequences, pairs, temporal overlaps |
| **Pivoting & Unpivoting** | 3 | Data reshaping, report generation |
| **Advanced Subqueries & CTEs** | 4 | Multi-step logic, correlated subqueries |
| **Set Operations & Combinatorics** | 3 | Intersections, exclusions, permutations |
| **Real-World Brain Teasers** | 3 | Interview classics with twists |

---

### 🎓 How to Master This Module

1. **Think in sets, not loops** - SQL is declarative; avoid imperative thinking
2. **Break down complexity** - Use CTEs to make logic readable
3. **Test edge cases** - Empty sets, NULL values, ties, duplicates
4. **Optimize from the start** - Window functions beat self-joins
5. **Explain your reasoning** - Interviewers care HOW you think

### 🏆 Interview Success Tips

✅ **Draw diagrams** - Visualize data relationships and transformations
✅ **Use CTEs for clarity** - Break complex queries into logical steps
✅ **Mention performance** - Explain why your solution scales
✅ **Test with sample data** - Walk through your logic with examples
✅ **Know alternatives** - "Here's another approach using..."

⚠️ **Red flags that fail interviews:**
- Saying "This can't be done in SQL" without trying
- Writing procedural code when set-based SQL would work
- Not considering NULL values or edge cases
- Ignoring performance implications (Cartesian products, etc.)
- Unable to explain your query logic step-by-step

---

**Ready to tackle the hardest SQL challenges? Let's dive in!** 🔥

## 📌 Section 1: Recursive Queries & Hierarchies (4 Questions)

Recursive CTEs unlock solutions for hierarchical data, graph traversal, and complex relationships.

### ❓ Question 1: Find All Direct and Indirect Reports

**Classic Interview Question:**
> "Given an employee table with employee_id, name, and manager_id, write a query to find ALL direct and indirect reports for a given manager (e.g., manager_id = 100), including their level in the hierarchy."

### ✅ Answer 1: Recursive Employee Hierarchy

#### **Problem Setup**

```sql
-- Sample employee table
CREATE TABLE employees (
  employee_id INT,
  name STRING,
  manager_id INT  -- NULL for CEO
);

INSERT INTO employees VALUES
  (1, 'Alice (CEO)', NULL),
  (2, 'Bob (VP)', 1),
  (3, 'Carol (VP)', 1),
  (4, 'Dave (Director)', 2),
  (5, 'Eve (Director)', 2),
  (6, 'Frank (Manager)', 4),
  (7, 'Grace (IC)', 6);
```

**Hierarchy:**
```
Alice (CEO)
├── Bob (VP)
│   ├── Dave (Director)
│   │   └── Frank (Manager)
│   │       └── Grace (IC)
│   └── Eve (Director)
└── Carol (VP)
```

---

#### **Solution: Recursive CTE**

```sql
WITH RECURSIVE reporting_chain AS (
  -- Base case: Start with the given manager
  SELECT 
    employee_id,
    name,
    manager_id,
    1 AS level,
    CAST(name AS STRING) AS hierarchy_path
  FROM employees
  WHERE employee_id = 2  -- Bob (starting point)
  
  UNION ALL
  
  -- Recursive case: Find direct reports
  SELECT 
    e.employee_id,
    e.name,
    e.manager_id,
    rc.level + 1,
    CONCAT(rc.hierarchy_path, ' > ', e.name) AS hierarchy_path
  FROM employees e
  INNER JOIN reporting_chain rc ON e.manager_id = rc.employee_id
)
SELECT 
  employee_id,
  name,
  level,
  hierarchy_path
FROM reporting_chain
ORDER BY level, name;
```

**Output:**
```
employee_id | name              | level | hierarchy_path
2           | Bob (VP)          | 1     | Bob (VP)
4           | Dave (Director)   | 2     | Bob (VP) > Dave (Director)
5           | Eve (Director)    | 2     | Bob (VP) > Eve (Director)
6           | Frank (Manager)   | 3     | Bob (VP) > Dave (Director) > Frank (Manager)
7           | Grace (IC)        | 4     | Bob (VP) > Dave (Director) > Frank (Manager) > Grace (IC)
```

---

#### **How Recursive CTEs Work**

**1. Base Case (Anchor):**
- Start with the "seed" rows (in this case, the manager)
- Sets level = 1

**2. Recursive Case:**
- JOIN the CTE to itself to find the next level
- Increment level + 1
- Build hierarchy path

**3. Termination:**
- Stops when no more rows match (leaf nodes reached)

---

#### **Variations & Follow-Up Questions**

**Variation 1: Find ALL Managers Above an Employee (Bottom-Up)**

```sql
WITH RECURSIVE manager_chain AS (
  -- Start with the employee
  SELECT employee_id, name, manager_id, 0 AS levels_up
  FROM employees
  WHERE employee_id = 7  -- Grace (IC)
  
  UNION ALL
  
  -- Traverse UP the hierarchy
  SELECT e.employee_id, e.name, e.manager_id, mc.levels_up + 1
  FROM employees e
  INNER JOIN manager_chain mc ON e.employee_id = mc.manager_id
)
SELECT * FROM manager_chain ORDER BY levels_up;
```

**Output:** Grace → Frank → Dave → Bob → Alice

---

**Variation 2: Find Total Headcount Per Manager**

```sql
WITH RECURSIVE reporting_chain AS (
  SELECT employee_id, name, manager_id, 1 AS level
  FROM employees
  
  UNION ALL
  
  SELECT e.employee_id, e.name, e.manager_id, rc.level + 1
  FROM employees e
  INNER JOIN reporting_chain rc ON e.manager_id = rc.employee_id
)
SELECT 
  e.employee_id,
  e.name AS manager_name,
  COUNT(rc.employee_id) - 1 AS total_reports  -- Subtract self
FROM employees e
LEFT JOIN reporting_chain rc ON rc.manager_id = e.employee_id OR rc.employee_id = e.employee_id
GROUP BY e.employee_id, e.name
ORDER BY total_reports DESC;
```

---

**Variation 3: Prevent Infinite Loops (Cycle Detection)**

```sql
WITH RECURSIVE reporting_chain AS (
  SELECT 
    employee_id, 
    name, 
    manager_id, 
    1 AS level,
    ARRAY(employee_id) AS visited_ids  -- Track visited nodes
  FROM employees
  WHERE employee_id = 2
  
  UNION ALL
  
  SELECT 
    e.employee_id, 
    e.name, 
    e.manager_id, 
    rc.level + 1,
    ARRAY_APPEND(rc.visited_ids, e.employee_id)
  FROM employees e
  INNER JOIN reporting_chain rc ON e.manager_id = rc.employee_id
  WHERE NOT ARRAY_CONTAINS(rc.visited_ids, e.employee_id)  -- Avoid cycles
    AND rc.level < 10  -- Max depth limit
)
SELECT * FROM reporting_chain;
```

---

#### **Performance Considerations**

**Optimization Tips:**

1. **Limit Recursion Depth:**
   ```sql
   WHERE rc.level < 10  -- Prevent runaway recursion
   ```

2. **Index Manager Column:**
   ```sql
   CREATE INDEX idx_manager ON employees(manager_id);
   ```

3. **Materialize for Repeated Queries:**
   ```sql
   -- Pre-compute hierarchy for large tables
   CREATE TABLE employee_hierarchy AS
   WITH RECURSIVE ... SELECT * FROM reporting_chain;
   ```

4. **Use Path Enumeration for Read-Heavy:**
   ```sql
   -- Store full path in table
   ALTER TABLE employees ADD COLUMN path STRING;
   -- path = '/1/2/4/6/7' for easy querying
   ```

---

#### **Common Pitfalls**

❌ **Forgetting to handle NULLs:**
```sql
-- CEO has manager_id = NULL, handle explicitly
WHERE manager_id IS NULL  -- For finding CEO
```

❌ **Infinite loops in bad data:**
```sql
-- If employee 5 reports to employee 7, and 7 reports to 5:
-- Add cycle detection or max depth limit
```

❌ **Cartesian explosion:**
```sql
-- Always JOIN on primary/foreign key relationship
INNER JOIN reporting_chain rc ON e.manager_id = rc.employee_id
```

---

#### **Interview Answer Template**

1. **Clarify requirements:** "Should I include the starting manager? What's the max depth?"
2. **Draw the hierarchy:** Sketch the org chart on whiteboard
3. **Explain recursive logic:** "Base case finds the root, recursive case traverses down"
4. **Walk through an example:** Show how Bob → Dave → Frank works
5. **Discuss performance:** "Index manager_id, limit depth, materialize for large tables"

In [0]:
%sql
-- Demo: Recursive Employee Hierarchy

-- Create sample employee table
CREATE OR REPLACE TABLE workspace.default.employees_hierarchy (
  employee_id INT,
  name STRING,
  manager_id INT,
  title STRING
);

INSERT INTO workspace.default.employees_hierarchy VALUES
  (1, 'Alice', NULL, 'CEO'),
  (2, 'Bob', 1, 'VP Engineering'),
  (3, 'Carol', 1, 'VP Sales'),
  (4, 'Dave', 2, 'Director'),
  (5, 'Eve', 2, 'Director'),
  (6, 'Frank', 4, 'Manager'),
  (7, 'Grace', 6, 'IC'),
  (8, 'Henry', 6, 'IC'),
  (9, 'Iris', 5, 'IC');

-- Query: Find all reports under Bob (employee_id = 2)
WITH RECURSIVE reporting_chain AS (
  -- Base case: Start with Bob
  SELECT 
    employee_id,
    name,
    manager_id,
    title,
    1 AS level,
    CAST(name AS STRING) AS hierarchy_path
  FROM workspace.default.employees_hierarchy
  WHERE employee_id = 2
  
  UNION ALL
  
  -- Recursive case: Find direct reports
  SELECT 
    e.employee_id,
    e.name,
    e.manager_id,
    e.title,
    rc.level + 1,
    CONCAT(rc.hierarchy_path, ' → ', e.name)
  FROM workspace.default.employees_hierarchy e
  INNER JOIN reporting_chain rc ON e.manager_id = rc.employee_id
)
SELECT 
  employee_id,
  name,
  title,
  level,
  hierarchy_path
FROM reporting_chain
ORDER BY level, name;

-- Query: Count total reports per manager
WITH RECURSIVE reporting_chain AS (
  SELECT employee_id, name, manager_id
  FROM workspace.default.employees_hierarchy
  
  UNION ALL
  
  SELECT e.employee_id, e.name, e.manager_id
  FROM workspace.default.employees_hierarchy e
  INNER JOIN reporting_chain rc ON e.manager_id = rc.employee_id
)
SELECT 
  e.name AS manager_name,
  COUNT(DISTINCT rc.employee_id) - 1 AS total_reports
FROM workspace.default.employees_hierarchy e
LEFT JOIN reporting_chain rc 
  ON rc.employee_id = e.employee_id OR rc.manager_id = e.employee_id
GROUP BY e.name
HAVING COUNT(DISTINCT rc.employee_id) > 1
ORDER BY total_reports DESC;

### ❓ Question 2: Bill of Materials - Explode Product Components

**Manufacturing/Supply Chain Interview:**
> "Given a parts table where products can be made of sub-assemblies (which are themselves products), write a query to explode a product into ALL its atomic components with quantities needed at each level."

### ✅ Answer 2: Bill of Materials Explosion

#### **Problem: Hierarchical Product Assembly**

```
Bicycle
├── Frame (1)
├── Wheel Assembly (2)
│   ├── Tire (1)
│   ├── Rim (1)
│   └── Spoke (36)
└── Chain (1)
```

**To make 1 Bicycle, you need:**
- 1 Frame
- 2 Tires (2 wheel assemblies * 1 tire each)
- 2 Rims
- 72 Spokes (2 wheel assemblies * 36 spokes each)
- 1 Chain

---

#### **Solution: Recursive BOM Explosion**

```sql
-- Parts table
CREATE TABLE parts (
  part_id STRING,
  part_name STRING,
  is_assembly BOOLEAN  -- TRUE if made of other parts
);

-- Bill of materials (which parts make up each assembly)
CREATE TABLE bom (
  parent_part_id STRING,
  child_part_id STRING,
  quantity INT
);

-- Recursive query to explode BOM
WITH RECURSIVE bom_explosion AS (
  -- Base case: Start with the top-level product
  SELECT 
    'BICYCLE' AS root_product,
    child_part_id AS part_id,
    quantity AS quantity_needed,
    1 AS level,
    CAST(child_part_id AS STRING) AS path
  FROM bom
  WHERE parent_part_id = 'BICYCLE'
  
  UNION ALL
  
  -- Recursive case: Explode sub-assemblies
  SELECT 
    be.root_product,
    b.child_part_id,
    be.quantity_needed * b.quantity,  -- Multiply quantities!
    be.level + 1,
    CONCAT(be.path, ' > ', b.child_part_id)
  FROM bom_explosion be
  INNER JOIN bom b ON be.part_id = b.parent_part_id
  INNER JOIN parts p ON be.part_id = p.part_id
  WHERE p.is_assembly = TRUE  -- Only explode assemblies
)
SELECT 
  part_id,
  SUM(quantity_needed) AS total_quantity,
  MAX(level) AS deepest_level,
  COLLECT_SET(path) AS assembly_paths
FROM bom_explosion
GROUP BY part_id
ORDER BY part_id;
```

**Output:**
```
part_id  | total_quantity | deepest_level | assembly_paths
FRAME    | 1              | 1             | ['FRAME']
WHEEL    | 2              | 1             | ['WHEEL']
TIRE     | 2              | 2             | ['WHEEL > TIRE']
RIM      | 2              | 2             | ['WHEEL > RIM']
SPOKE    | 72             | 2             | ['WHEEL > SPOKE']
CHAIN    | 1              | 1             | ['CHAIN']
```

---

#### **Key Insights**

1. **Multiply Quantities:** `be.quantity_needed * b.quantity` at each level
2. **Filter Assemblies:** Only recurse into `is_assembly = TRUE` parts
3. **Sum Across Paths:** Same part may appear in multiple branches
4. **Track Paths:** Helps debug and visualize the BOM

---

#### **Interview Follow-Up: Cost Rollup**

```sql
-- Calculate total cost to build a product
WITH RECURSIVE bom_explosion AS (
  -- ... same as above ...
),
atomic_parts AS (
  SELECT part_id, SUM(quantity_needed) AS total_quantity
  FROM bom_explosion
  WHERE part_id NOT IN (SELECT DISTINCT parent_part_id FROM bom)  -- Leaf nodes only
  GROUP BY part_id
)
SELECT 
  SUM(ap.total_quantity * p.unit_cost) AS total_product_cost
FROM atomic_parts ap
JOIN parts p ON ap.part_id = p.part_id;
```

### ❓ Question 3: Find Shortest Path Between Two Nodes

**Graph/Network Interview Question:**
> "Given a table of city connections with distances, find the shortest path from City A to City Z. Return the path and total distance."

### ✅ Answer 3: Shortest Path with Recursive CTE

#### **Problem: Find Shortest Route**

```
Connections:
A --10-- B --5-- C
|        |       |
15      20      10
|        |       |
D --5--- E --2-- Z
```

**Find shortest path:** A → Z

---

#### **Solution: Dijkstra-Style Recursive Query**

```sql
CREATE TABLE connections (
  from_city STRING,
  to_city STRING,
  distance INT
);

INSERT INTO connections VALUES
  ('A', 'B', 10), ('B', 'A', 10),  -- Bidirectional
  ('A', 'D', 15), ('D', 'A', 15),
  ('B', 'C', 5), ('C', 'B', 5),
  ('B', 'E', 20), ('E', 'B', 20),
  ('C', 'Z', 10), ('Z', 'C', 10),
  ('D', 'E', 5), ('E', 'D', 5),
  ('E', 'Z', 2), ('Z', 'E', 2);

WITH RECURSIVE paths AS (
  -- Base case: Start from city A
  SELECT 
    from_city,
    to_city,
    distance AS total_distance,
    CAST(CONCAT(from_city, ' → ', to_city) AS STRING) AS path,
    ARRAY(from_city, to_city) AS visited
  FROM connections
  WHERE from_city = 'A'
  
  UNION ALL
  
  -- Recursive case: Extend paths
  SELECT 
    p.from_city,
    c.to_city,
    p.total_distance + c.distance,
    CONCAT(p.path, ' → ', c.to_city),
    ARRAY_UNION(p.visited, ARRAY(c.to_city))
  FROM paths p
  INNER JOIN connections c ON p.to_city = c.from_city
  WHERE NOT ARRAY_CONTAINS(p.visited, c.to_city)  -- Avoid cycles
    AND p.total_distance + c.distance < 50  -- Prune expensive paths
)
SELECT 
  path,
  total_distance
FROM paths
WHERE to_city = 'Z'
ORDER BY total_distance
LIMIT 1;
```

**Output:**
```
path               | total_distance
A → D → E → Z    | 22
```

**All paths found (for comparison):**
```
A → B → C → Z      | 25
A → B → E → Z      | 32
A → D → E → Z      | 22  ← Shortest!
```

---

#### **Optimizations for Large Graphs**

1. **Prune Expensive Paths Early:**
   ```sql
   WHERE p.total_distance + c.distance < (
     SELECT MIN(total_distance) * 1.5 FROM paths WHERE to_city = 'Z'
   )
   ```

2. **Use Priority Queue (External Algorithm):**
   - For very large graphs, implement Dijkstra in Python with Spark
   - SQL recursion is elegant but not optimal for huge networks

3. **Pre-compute Distances:**
   ```sql
   -- Materialize all-pairs shortest paths (Floyd-Warshall)
   CREATE TABLE shortest_paths AS ...
   ```

## 📌 Section 2: Advanced Window Functions (5 Questions)

Window functions solve complex analytics problems that would otherwise require self-joins or subqueries.

### ❓ Question 4: Find Consecutive Date Ranges (Gaps and Islands)

**Classic SQL Puzzle:**
> "Given a table of user login dates, identify consecutive login streaks. For example, if a user logged in on Jan 1, 2, 3, 5, 6, that's 2 streaks: Jan 1-3 (3 days) and Jan 5-6 (2 days)."

### ✅ Answer 4: Gaps and Islands Pattern

#### **The "Gaps and Islands" Problem**

**Islands** = Consecutive sequences (login streaks)
**Gaps** = Missing dates (days not logged in)

---

#### **Solution: ROW_NUMBER Trick**

**Key Insight:** For consecutive dates, `date - ROW_NUMBER()` is constant!

```
Date       | ROW_NUMBER | date - ROW_NUMBER | Island Group
2024-01-01 | 1          | 2023-12-31        | A
2024-01-02 | 2          | 2023-12-31        | A  ← Same!
2024-01-03 | 3          | 2023-12-31        | A  ← Same!
2024-01-05 | 4          | 2024-01-01        | B  ← New group
2024-01-06 | 5          | 2024-01-01        | B  ← Same!
```

**Full Query:**

```sql
WITH login_dates AS (
  SELECT DISTINCT user_id, login_date
  FROM user_logins
),
island_groups AS (
  SELECT 
    user_id,
    login_date,
    DATE_SUB(login_date, ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY login_date)) AS island_id
  FROM login_dates
)
SELECT 
  user_id,
  MIN(login_date) AS streak_start,
  MAX(login_date) AS streak_end,
  COUNT(*) AS streak_length
FROM island_groups
GROUP BY user_id, island_id
HAVING COUNT(*) >= 3  -- Only streaks of 3+ days
ORDER BY user_id, streak_start;
```

**Output:**
```
user_id | streak_start | streak_end | streak_length
1       | 2024-01-01   | 2024-01-03 | 3
1       | 2024-01-10   | 2024-01-15 | 6
2       | 2024-01-05   | 2024-01-08 | 4
```

---

#### **Find Gaps (Missing Dates)**

```sql
WITH login_dates AS (
  SELECT DISTINCT login_date
  FROM user_logins
  WHERE user_id = 123
),
date_diffs AS (
  SELECT 
    login_date AS current_date,
    LAG(login_date) OVER (ORDER BY login_date) AS previous_date,
    DATEDIFF(login_date, LAG(login_date) OVER (ORDER BY login_date)) AS gap_days
  FROM login_dates
)
SELECT 
  previous_date AS gap_start,
  current_date AS gap_end,
  gap_days - 1 AS missing_days
FROM date_diffs
WHERE gap_days > 1  -- Gaps only (not consecutive)
ORDER BY gap_start;
```

---

#### **Interview Variations**

**Find Longest Streak Per User:**
```sql
WITH streaks AS (
  -- ... island_groups CTE from above ...
  SELECT user_id, COUNT(*) AS streak_length
  FROM island_groups
  GROUP BY user_id, island_id
)
SELECT 
  user_id,
  MAX(streak_length) AS longest_streak
FROM streaks
GROUP BY user_id;
```

**Find Users with No Gaps in Last 30 Days:**
```sql
WITH expected_dates AS (
  SELECT EXPLODE(SEQUENCE(CURRENT_DATE - 29, CURRENT_DATE)) AS date
),
actual_logins AS (
  SELECT DISTINCT login_date FROM user_logins WHERE user_id = 123
)
SELECT COUNT(*) AS perfect_attendance
FROM expected_dates e
INNER JOIN actual_logins a ON e.date = a.login_date
HAVING COUNT(*) = 30;  -- All 30 days present
```

In [0]:
%sql
-- Demo: Find Login Streaks (Gaps and Islands)

-- Sample login data
CREATE OR REPLACE TABLE workspace.default.user_logins (
  user_id INT,
  login_date DATE
);

INSERT INTO workspace.default.user_logins VALUES
  (1, '2024-01-01'), (1, '2024-01-02'), (1, '2024-01-03'),  -- Streak 1
  (1, '2024-01-05'), (1, '2024-01-06'),                     -- Streak 2 (gap on Jan 4)
  (1, '2024-01-10'), (1, '2024-01-11'), (1, '2024-01-12'), (1, '2024-01-13'),  -- Streak 3
  (2, '2024-01-01'), (2, '2024-01-02'), (2, '2024-01-04'), (2, '2024-01-05');  -- User 2

-- Find consecutive login streaks
WITH login_dates AS (
  SELECT DISTINCT user_id, login_date
  FROM workspace.default.user_logins
),
island_groups AS (
  SELECT 
    user_id,
    login_date,
    DATE_SUB(login_date, ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY login_date)) AS island_id
  FROM login_dates
)
SELECT 
  user_id,
  MIN(login_date) AS streak_start,
  MAX(login_date) AS streak_end,
  COUNT(*) AS streak_length,
  CONCAT(MIN(login_date), ' to ', MAX(login_date)) AS streak_period
FROM island_groups
GROUP BY user_id, island_id
ORDER BY user_id, streak_start;

-- Find gaps (missing dates)
WITH date_diffs AS (
  SELECT 
    user_id,
    login_date,
    LAG(login_date) OVER (PARTITION BY user_id ORDER BY login_date) AS prev_date,
    DATEDIFF(login_date, LAG(login_date) OVER (PARTITION BY user_id ORDER BY login_date)) AS gap_size
  FROM workspace.default.user_logins
)
SELECT 
  user_id,
  prev_date AS gap_start,
  login_date AS gap_end,
  gap_size - 1 AS missing_days
FROM date_diffs
WHERE gap_size > 1
ORDER BY user_id, prev_date;

### ❓ Question 5: Running Total with Reset on Condition

**Advanced Analytics Question:**
> "Given a sales table, calculate a running total of sales amount, but RESET the running total to zero whenever a refund occurs. Show the running balance for each transaction."

### ✅ Answer 5: Running Total with Conditional Reset

#### **Problem: Account Balance with Reset Events**

```
Transactions:
Date       | Type    | Amount | Expected Balance
2024-01-01 | Sale    | 100    | 100
2024-01-02 | Sale    | 50     | 150
2024-01-03 | Refund  | -150   | 0    ← Reset!
2024-01-04 | Sale    | 200    | 200  ← Start fresh
2024-01-05 | Sale    | 100    | 300
```

---

#### **Solution: Two-Step Window Functions**

```sql
WITH transactions AS (
  SELECT 
    transaction_date,
    transaction_type,
    amount,
    -- Step 1: Identify reset points (cumulative count of resets)
    SUM(CASE WHEN transaction_type = 'Refund' THEN 1 ELSE 0 END) 
      OVER (ORDER BY transaction_date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS reset_group
  FROM sales_transactions
)
SELECT 
  transaction_date,
  transaction_type,
  amount,
  -- Step 2: Running total within each reset group
  SUM(amount) OVER (
    PARTITION BY reset_group 
    ORDER BY transaction_date 
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
  ) AS running_balance
FROM transactions
ORDER BY transaction_date;
```

**Output:**
```
transaction_date | transaction_type | amount | running_balance
2024-01-01       | Sale            | 100    | 100
2024-01-02       | Sale            | 50     | 150
2024-01-03       | Refund          | -150   | 0
2024-01-04       | Sale            | 200    | 200
2024-01-05       | Sale            | 100    | 300
```

---

#### **How It Works:**

**Step 1: Create Reset Groups**
```sql
SUM(CASE WHEN transaction_type = 'Refund' THEN 1 ELSE 0 END) 
  OVER (ORDER BY transaction_date)
```

This creates a "group ID" that increments after each refund:
```
Date       | Type   | Amount | reset_group
2024-01-01 | Sale   | 100    | 0
2024-01-02 | Sale   | 50     | 0
2024-01-03 | Refund | -150   | 1  ← Incremented
2024-01-04 | Sale   | 200    | 1
2024-01-05 | Sale   | 100    | 1
```

**Step 2: Running Total Per Group**
```sql
SUM(amount) OVER (PARTITION BY reset_group ORDER BY transaction_date)
```

Calculates running total WITHIN each group separately.

---

#### **Variation: Reset on Negative Balance**

```sql
WITH balances AS (
  SELECT 
    *,
    SUM(amount) OVER (ORDER BY transaction_date ROWS UNBOUNDED PRECEDING) AS raw_balance
  FROM transactions
),
reset_flags AS (
  SELECT 
    *,
    CASE WHEN raw_balance < 0 THEN 1 ELSE 0 END AS is_negative,
    SUM(CASE WHEN raw_balance < 0 THEN 1 ELSE 0 END) 
      OVER (ORDER BY transaction_date ROWS UNBOUNDED PRECEDING) AS reset_group
  FROM balances
)
SELECT 
  transaction_date,
  amount,
  SUM(CASE WHEN is_negative = 0 THEN amount ELSE 0 END) 
    OVER (PARTITION BY reset_group ORDER BY transaction_date) AS safe_balance
FROM reset_flags;
```

---

#### **Interview Variation: Cumulative Sum Until Threshold**

**Problem:** Running total, but reset when it exceeds 1000

```sql
WITH recursive_balance AS (
  SELECT 
    transaction_id,
    amount,
    CASE 
      WHEN SUM(amount) OVER (ORDER BY transaction_id ROWS UNBOUNDED PRECEDING) > 1000 
      THEN 1 
      ELSE 0 
    END AS exceeds_threshold
  FROM transactions
)
SELECT 
  transaction_id,
  amount,
  SUM(CASE WHEN exceeds_threshold = 0 THEN amount ELSE 0 END) 
    OVER (PARTITION BY exceeds_threshold ORDER BY transaction_id) AS capped_balance
FROM recursive_balance;
```

### ❓ Question 6: Calculate Median Without Using PERCENTILE_CONT

**Tricky Interview Question:**
> "Calculate the median salary for each department WITHOUT using PERCENTILE_CONT, MEDIAN, or any built-in percentile functions. Use only basic window functions."

### ✅ Answer 6: Median Using ROW_NUMBER

#### **Median Definition**

- **Odd count:** Middle value (position = (N+1)/2)
- **Even count:** Average of two middle values (positions = N/2 and N/2+1)

---

#### **Solution: Window Functions Only**

```sql
WITH ranked_salaries AS (
  SELECT 
    department,
    salary,
    ROW_NUMBER() OVER (PARTITION BY department ORDER BY salary) AS row_asc,
    ROW_NUMBER() OVER (PARTITION BY department ORDER BY salary DESC) AS row_desc,
    COUNT(*) OVER (PARTITION BY department) AS total_count
  FROM employees
)
SELECT 
  department,
  AVG(salary) AS median_salary
FROM ranked_salaries
WHERE 
  -- For odd count: row_asc = row_desc (middle value)
  -- For even count: row_asc IN (N/2, N/2+1)
  row_asc IN (FLOOR((total_count + 1) / 2.0), CEIL((total_count + 1) / 2.0))
  OR row_asc = row_desc  -- Alternative: middle element
GROUP BY department;
```

**Alternative (cleaner):**

```sql
WITH ranked AS (
  SELECT 
    department,
    salary,
    ROW_NUMBER() OVER (PARTITION BY department ORDER BY salary) AS rn,
    COUNT(*) OVER (PARTITION BY department) AS cnt
  FROM employees
)
SELECT 
  department,
  AVG(salary) AS median_salary
FROM ranked
WHERE rn BETWEEN cnt/2.0 AND cnt/2.0 + 1
GROUP BY department;
```

**How it works:**

For **odd count** (e.g., 5 rows):
- Middle = row 3
- `BETWEEN 2.5 AND 3.5` captures row 3 only
- `AVG(salary)` of 1 row = that salary

For **even count** (e.g., 6 rows):
- Middle = rows 3 and 4
- `BETWEEN 3.0 AND 4.0` captures rows 3 and 4
- `AVG(salary)` of 2 rows = median

---

#### **Example Walkthrough**

```sql
-- Sample data
Department | Salary | rn | cnt
Eng        | 50k    | 1  | 5
Eng        | 60k    | 2  | 5
Eng        | 70k    | 3  | 5  ← Median (odd count)
Eng        | 80k    | 4  | 5
Eng        | 90k    | 5  | 5

Sales      | 40k    | 1  | 4
Sales      | 50k    | 2  | 4  ← Median pair
Sales      | 60k    | 3  | 4  ← Median pair
Sales      | 70k    | 4  | 4

-- Engineering: rn BETWEEN 2.5 AND 3.5 → row 3 → 70k
-- Sales: rn BETWEEN 2.0 AND 3.0 → rows 2,3 → AVG(50k, 60k) = 55k
```

---

#### **Alternative: Using Ascending/Descending Ranks**

```sql
WITH ranked AS (
  SELECT 
    department,
    salary,
    ROW_NUMBER() OVER (PARTITION BY department ORDER BY salary ASC) AS rn_asc,
    ROW_NUMBER() OVER (PARTITION BY department ORDER BY salary DESC) AS rn_desc
  FROM employees
)
SELECT 
  department,
  AVG(salary) AS median_salary
FROM ranked
WHERE ABS(rn_asc - rn_desc) <= 1  -- Middle element(s)
GROUP BY department;
```

**Why this works:**
- For **odd count**, the middle element has `rn_asc = rn_desc`
- For **even count**, the two middle elements have `|rn_asc - rn_desc| = 1`

---

#### **Interview Follow-Up: Mode (Most Frequent Value)**

```sql
WITH value_counts AS (
  SELECT 
    department,
    salary,
    COUNT(*) AS frequency,
    RANK() OVER (PARTITION BY department ORDER BY COUNT(*) DESC) AS freq_rank
  FROM employees
  GROUP BY department, salary
)
SELECT 
  department,
  salary AS mode_salary,
  frequency
FROM value_counts
WHERE freq_rank = 1;
```

### ❓ Question 7: Find First Non-NULL Value Per Group

**Data Quality Interview Question:**
> "Given a time-series table where some values are NULL, find the FIRST non-NULL value for each sensor, ordered by timestamp."

### ✅ Answer 7: First Non-NULL Value

#### **Problem: Skip NULLs in Window Functions**

```
Sensor | Timestamp  | Temperature
A      | 10:00      | NULL
A      | 10:05      | NULL
A      | 10:10      | 72.5  ← First non-NULL!
A      | 10:15      | 73.0
B      | 10:00      | 68.0  ← First non-NULL!
B      | 10:05      | NULL
```

---

#### **Solution 1: Filter then FIRST_VALUE**

```sql
WITH non_null_values AS (
  SELECT 
    sensor_id,
    timestamp,
    temperature
  FROM sensor_readings
  WHERE temperature IS NOT NULL
)
SELECT 
  sensor_id,
  FIRST_VALUE(temperature) OVER (
    PARTITION BY sensor_id 
    ORDER BY timestamp
  ) AS first_non_null_temp
FROM non_null_values;
```

---

#### **Solution 2: IGNORE NULLS (Databricks/Spark SQL)**

```sql
SELECT 
  sensor_id,
  timestamp,
  temperature,
  FIRST_VALUE(temperature IGNORE NULLS) OVER (
    PARTITION BY sensor_id 
    ORDER BY timestamp
  ) AS first_non_null_temp
FROM sensor_readings;
```

**Note:** `IGNORE NULLS` skips NULL values in window function evaluation.

---

#### **Solution 3: Self-Join to First Non-NULL**

```sql
WITH first_non_null AS (
  SELECT 
    sensor_id,
    MIN(timestamp) AS first_valid_time
  FROM sensor_readings
  WHERE temperature IS NOT NULL
  GROUP BY sensor_id
)
SELECT 
  s.sensor_id,
  s.timestamp,
  s.temperature,
  f.first_valid_time,
  sf.temperature AS first_non_null_value
FROM sensor_readings s
JOIN first_non_null f ON s.sensor_id = f.sensor_id
JOIN sensor_readings sf ON f.sensor_id = sf.sensor_id AND f.first_valid_time = sf.timestamp;
```

---

#### **Variation: Forward Fill (Propagate Last Known Value)**

```sql
SELECT 
  sensor_id,
  timestamp,
  temperature,
  -- Forward fill: Use last non-NULL value
  LAST_VALUE(temperature IGNORE NULLS) OVER (
    PARTITION BY sensor_id 
    ORDER BY timestamp
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
  ) AS filled_temperature
FROM sensor_readings;
```

**Output:**
```
Sensor | Timestamp | Temperature | filled_temperature
A      | 10:00     | NULL        | NULL
A      | 10:05     | NULL        | NULL
A      | 10:10     | 72.5        | 72.5
A      | 10:15     | NULL        | 72.5  ← Forward filled!
A      | 10:20     | 73.0        | 73.0
A      | 10:25     | NULL        | 73.0  ← Forward filled!
```

---

#### **Interview Follow-Up: Backward Fill**

```sql
SELECT 
  sensor_id,
  timestamp,
  temperature,
  -- Look AHEAD for next non-NULL
  FIRST_VALUE(temperature IGNORE NULLS) OVER (
    PARTITION BY sensor_id 
    ORDER BY timestamp
    ROWS BETWEEN CURRENT ROW AND UNBOUNDED FOLLOWING
  ) AS backfilled_temperature
FROM sensor_readings;
```

## 📌 Section 3: Self-Joins & Complex Relationships (3 Questions)

Self-joins solve problems involving pairs, sequences, and temporal overlaps.

### ❓ Question 8: Find Overlapping Time Intervals

**Calendar/Scheduling Interview Question:**
> "Given a table of meeting room bookings with start_time and end_time, find all pairs of bookings that overlap (conflict). A conflict exists when two bookings use the same room and their time ranges overlap."

### ✅ Answer 8: Temporal Overlap Detection

#### **Overlap Condition**

Two intervals overlap if:
```
A.start < B.end  AND  A.end > B.start
```

**Visual:**
```
A: |-------|        Overlaps with B, C
B:     |-------|    Overlaps with A, C
C:         |---|    Overlaps with A, B
D:              |---| No overlap
```

---

#### **Solution: Self-Join**

```sql
SELECT 
  a.booking_id AS booking_1,
  b.booking_id AS booking_2,
  a.room_id,
  a.start_time AS booking_1_start,
  a.end_time AS booking_1_end,
  b.start_time AS booking_2_start,
  b.end_time AS booking_2_end
FROM bookings a
JOIN bookings b 
  ON a.room_id = b.room_id  -- Same room
  AND a.booking_id < b.booking_id  -- Avoid duplicates (A-B and B-A)
  AND a.start_time < b.end_time  -- Overlap condition
  AND a.end_time > b.start_time
ORDER BY a.room_id, a.start_time;
```

**Output:**
```
booking_1 | booking_2 | room_id | booking_1_start | booking_1_end | booking_2_start | booking_2_end
1         | 2         | 101     | 10:00           | 11:00         | 10:30           | 11:30
2         | 3         | 101     | 10:30           | 11:30         | 11:00           | 12:00
```

---

#### **Find Rooms with NO Conflicts**

```sql
WITH conflicts AS (
  SELECT DISTINCT a.room_id
  FROM bookings a
  JOIN bookings b 
    ON a.room_id = b.room_id
    AND a.booking_id < b.booking_id
    AND a.start_time < b.end_time
    AND a.end_time > b.start_time
)
SELECT DISTINCT room_id
FROM bookings
WHERE room_id NOT IN (SELECT room_id FROM conflicts);
```

---

#### **Find Maximum Concurrent Bookings**

```sql
WITH time_points AS (
  SELECT start_time AS time_point, room_id, 1 AS change
  FROM bookings
  UNION ALL
  SELECT end_time AS time_point, room_id, -1 AS change
  FROM bookings
)
SELECT 
  room_id,
  MAX(concurrent_bookings) AS max_concurrent
FROM (
  SELECT 
    room_id,
    time_point,
    SUM(change) OVER (PARTITION BY room_id ORDER BY time_point) AS concurrent_bookings
  FROM time_points
) 
GROUP BY room_id;
```

---

#### **Interview Follow-Up: Gap Detection**

**Find available time slots between bookings:**

```sql
SELECT 
  room_id,
  LAG(end_time) OVER (PARTITION BY room_id ORDER BY start_time) AS gap_start,
  start_time AS gap_end,
  TIMESTAMPDIFF(MINUTE, 
    LAG(end_time) OVER (PARTITION BY room_id ORDER BY start_time),
    start_time
  ) AS gap_minutes
FROM bookings
WHERE LAG(end_time) OVER (PARTITION BY room_id ORDER BY start_time) < start_time
ORDER BY room_id, gap_start;
```

### ❓ Question 9: Identify Pairs of Consecutive Transactions

**E-commerce Interview Question:**
> "Find all pairs of transactions where the same customer made two purchases within 24 hours. Return customer_id, first transaction, second transaction, and time difference."

### ✅ Answer 9: Consecutive Transaction Pairs

#### **Solution: LEAD Window Function**

```sql
WITH next_purchase AS (
  SELECT 
    customer_id,
    transaction_id,
    transaction_time,
    amount,
    LEAD(transaction_id) OVER (PARTITION BY customer_id ORDER BY transaction_time) AS next_txn_id,
    LEAD(transaction_time) OVER (PARTITION BY customer_id ORDER BY transaction_time) AS next_txn_time,
    LEAD(amount) OVER (PARTITION BY customer_id ORDER BY transaction_time) AS next_amount
  FROM transactions
)
SELECT 
  customer_id,
  transaction_id AS first_txn,
  next_txn_id AS second_txn,
  transaction_time AS first_time,
  next_txn_time AS second_time,
  TIMESTAMPDIFF(HOUR, transaction_time, next_txn_time) AS hours_between,
  amount + next_amount AS total_spent
FROM next_purchase
WHERE next_txn_time IS NOT NULL
  AND TIMESTAMPDIFF(HOUR, transaction_time, next_txn_time) <= 24
ORDER BY customer_id, transaction_time;
```

---

#### **Variation: Find Triplets (3 Consecutive)**

```sql
WITH enriched AS (
  SELECT 
    customer_id,
    transaction_id AS txn1,
    LEAD(transaction_id, 1) OVER w AS txn2,
    LEAD(transaction_id, 2) OVER w AS txn3,
    transaction_time AS time1,
    LEAD(transaction_time, 1) OVER w AS time2,
    LEAD(transaction_time, 2) OVER w AS time3
  FROM transactions
  WINDOW w AS (PARTITION BY customer_id ORDER BY transaction_time)
)
SELECT 
  customer_id,
  txn1, txn2, txn3,
  TIMESTAMPDIFF(HOUR, time1, time3) AS total_hours
FROM enriched
WHERE txn3 IS NOT NULL
  AND TIMESTAMPDIFF(HOUR, time1, time3) <= 48;
```

---

#### **Self-Join Alternative (Less Efficient)**

```sql
SELECT 
  t1.customer_id,
  t1.transaction_id AS first_txn,
  t2.transaction_id AS second_txn,
  TIMESTAMPDIFF(HOUR, t1.transaction_time, t2.transaction_time) AS hours_between
FROM transactions t1
JOIN transactions t2 
  ON t1.customer_id = t2.customer_id
  AND t1.transaction_time < t2.transaction_time
  AND TIMESTAMPDIFF(HOUR, t1.transaction_time, t2.transaction_time) <= 24
  AND NOT EXISTS (
    SELECT 1 FROM transactions t3
    WHERE t3.customer_id = t1.customer_id
      AND t3.transaction_time > t1.transaction_time
      AND t3.transaction_time < t2.transaction_time
  )
ORDER BY t1.customer_id, t1.transaction_time;
```

## 📌 Section 4: Pivoting & Unpivoting (3 Questions)

Reshaping data between long and wide formats is essential for reporting and analytics.

### ❓ Question 10: Pivot Sales by Month (Dynamic Columns)

**Reporting Interview Question:**
> "Given a sales table with (product, month, amount), pivot it so each month becomes a column. The challenge: the number of months is not fixed—handle it dynamically."

### ✅ Answer 10: Dynamic PIVOT

#### **Input Data (Long Format):**
```
product | month   | sales
Laptop  | Jan     | 1000
Laptop  | Feb     | 1200
Mouse   | Jan     | 100
Mouse   | Feb     | 150
```

**Desired Output (Wide Format):**
```
product | Jan  | Feb  | Mar  | ...
Laptop  | 1000 | 1200 | NULL
Mouse   | 100  | 150  | NULL
```

---

#### **Solution 1: PIVOT Clause (Spark SQL 3.4+)**

```sql
SELECT *
FROM (
  SELECT product, month, sales
  FROM sales_data
)
PIVOT (
  SUM(sales)
  FOR month IN ('Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun')
);
```

**Problem:** Hardcoded month list!

---

#### **Solution 2: CASE Statements (Dynamic)**

```sql
SELECT 
  product,
  SUM(CASE WHEN month = 'Jan' THEN sales ELSE 0 END) AS Jan,
  SUM(CASE WHEN month = 'Feb' THEN sales ELSE 0 END) AS Feb,
  SUM(CASE WHEN month = 'Mar' THEN sales ELSE 0 END) AS Mar,
  SUM(CASE WHEN month = 'Apr' THEN sales ELSE 0 END) AS Apr,
  SUM(CASE WHEN month = 'May' THEN sales ELSE 0 END) AS May,
  SUM(CASE WHEN month = 'Jun' THEN sales ELSE 0 END) AS Jun
FROM sales_data
GROUP BY product;
```

**Problem:** Still hardcoded!

---

#### **Solution 3: Truly Dynamic (Python + SQL)**

```python
# Step 1: Get distinct months dynamically
months = spark.sql("""
  SELECT DISTINCT month 
  FROM sales_data 
  ORDER BY month
""").rdd.flatMap(lambda x: x).collect()

# Step 2: Build CASE statements dynamically
case_statements = ",\n  ".join([
  f"SUM(CASE WHEN month = '{m}' THEN sales ELSE 0 END) AS `{m}`"
  for m in months
])

# Step 3: Execute dynamic SQL
query = f"""
SELECT 
  product,
  {case_statements}
FROM sales_data
GROUP BY product
"""

result_df = spark.sql(query)
display(result_df)
```

---

#### **Solution 4: Using groupBy + pivot (PySpark)**

```python
# Most elegant!
from pyspark.sql import functions as F

pivoted_df = (
  spark.table("sales_data")
    .groupBy("product")
    .pivot("month")  # Automatically discovers distinct months
    .agg(F.sum("sales"))
)

display(pivoted_df)
```

---

#### **Unpivot (Reverse Operation)**

**Input (Wide):**
```
product | Jan | Feb | Mar
Laptop  | 100 | 200 | 300
```

**Output (Long):**
```
product | month | sales
Laptop  | Jan   | 100
Laptop  | Feb   | 200
Laptop  | Mar   | 300
```

**Solution:**

```sql
-- Spark SQL 3.4+
SELECT product, month, sales
FROM sales_wide
UNPIVOT (
  sales FOR month IN (Jan, Feb, Mar, Apr, May, Jun)
);
```

**Alternative (STACK function):**

```sql
SELECT 
  product,
  month,
  sales
FROM sales_wide
LATERAL VIEW STACK(6,
  'Jan', Jan,
  'Feb', Feb,
  'Mar', Mar,
  'Apr', Apr,
  'May', May,
  'Jun', Jun
) AS month, sales;
```

---

#### **Interview Pro Tip**

**Question:** "How would you handle 100 months?"

**Answer:** "I'd use PySpark's `.pivot()` which auto-discovers columns, or generate CASE statements programmatically. For reporting, I'd also question whether a pivot is the right approach—often a grouped bar chart on long-format data is more maintainable."

## 📌 Section 5: SQL Brain Teasers (5 Questions)

Classic interview puzzles that test creative problem-solving and deep SQL knowledge.

### ❓ Question 11: Find Nth Highest Salary

**Classic Interview Question:**
> "Write a query to find the second highest salary. If there's no second highest (e.g., all salaries are the same or only one employee), return NULL."

### ✅ Answer 11: Nth Highest Salary (Multiple Solutions)

#### **Solution 1: DENSE_RANK (Best)**

```sql
WITH ranked_salaries AS (
  SELECT 
    salary,
    DENSE_RANK() OVER (ORDER BY salary DESC) AS rank
  FROM employees
)
SELECT salary AS second_highest_salary
FROM ranked_salaries
WHERE rank = 2
LIMIT 1;
```

**Why DENSE_RANK?**
- `DENSE_RANK`: 100, 100, 90, 80 → ranks 1, 1, 2, 3 ✅
- `RANK`: 100, 100, 90, 80 → ranks 1, 1, 3, 4 ❌ (skips rank 2!)
- `ROW_NUMBER`: 100, 100, 90, 80 → ranks 1, 2, 3, 4 ❌ (treats ties as different)

---

#### **Solution 2: Subquery with LIMIT/OFFSET**

```sql
SELECT 
  (SELECT DISTINCT salary 
   FROM employees 
   ORDER BY salary DESC 
   LIMIT 1 OFFSET 1) AS second_highest_salary;
```

**Returns NULL if no second salary exists!**

---

#### **Solution 3: MAX with Subquery**

```sql
SELECT MAX(salary) AS second_highest_salary
FROM employees
WHERE salary < (SELECT MAX(salary) FROM employees);
```

**Problem:** Doesn't generalize well to "Nth" highest.

---

#### **Solution 4: Self-Join (Inefficient)**

```sql
SELECT e1.salary
FROM employees e1
WHERE 1 = (
  SELECT COUNT(DISTINCT e2.salary)
  FROM employees e2
  WHERE e2.salary > e1.salary
)
LIMIT 1;
```

**Explanation:** Find salary where exactly 1 distinct salary is higher.

---

#### **Generalized: Nth Highest Salary (Function)**

```sql
CREATE FUNCTION getNthHighestSalary(N INT)
RETURNS TABLE(salary DECIMAL(10,2))
RETURN
  WITH ranked AS (
    SELECT 
      salary,
      DENSE_RANK() OVER (ORDER BY salary DESC) AS rank
    FROM employees
  )
  SELECT DISTINCT salary
  FROM ranked
  WHERE rank = N;

-- Usage
SELECT * FROM getNthHighestSalary(3);  -- 3rd highest
```

---

#### **Edge Cases to Handle**

1. **No second highest (only 1 employee):** Return NULL
2. **All salaries the same:** Return NULL
3. **Ties:** 100, 100, 90 → second highest is 90 (not 100)

```sql
-- Robust solution with COALESCE
SELECT COALESCE(
  (SELECT DISTINCT salary 
   FROM employees 
   ORDER BY salary DESC 
   LIMIT 1 OFFSET 1),
  NULL
) AS second_highest_salary;
```

### ❓ Question 12: Remove Duplicate Rows (Keep First)

**Data Cleaning Interview Question:**
> "Given a table with duplicate rows (based on email), delete all duplicates keeping only the row with the smallest ID."

### ✅ Answer 12: Delete Duplicates with ROW_NUMBER

#### **Problem Setup**

```sql
CREATE TABLE users (
  id INT,
  email STRING,
  name STRING
);

INSERT INTO users VALUES
  (1, 'alice@email.com', 'Alice'),
  (2, 'bob@email.com', 'Bob'),
  (3, 'alice@email.com', 'Alice Duplicate'),  -- Duplicate!
  (4, 'carol@email.com', 'Carol'),
  (5, 'alice@email.com', 'Alice Triplicate');  -- Duplicate!
```

**Goal:** Keep only ID=1 for alice@email.com, delete ID=3 and ID=5.

---

#### **Solution 1: ROW_NUMBER + DELETE**

```sql
-- Step 1: Identify duplicates
WITH ranked AS (
  SELECT 
    id,
    ROW_NUMBER() OVER (PARTITION BY email ORDER BY id) AS rn
  FROM users
)
-- Step 2: Delete where row number > 1
DELETE FROM users
WHERE id IN (
  SELECT id FROM ranked WHERE rn > 1
);
```

---

#### **Solution 2: Create Deduplicated Table (Databricks)**

```sql
-- Databricks doesn't support DELETE with CTE, use MERGE
MERGE INTO users AS target
USING (
  SELECT id, email, name,
    ROW_NUMBER() OVER (PARTITION BY email ORDER BY id) AS rn
  FROM users
) AS source
ON target.id = source.id AND source.rn > 1
WHEN MATCHED THEN DELETE;
```

---

#### **Solution 3: CREATE TABLE AS SELECT (Best for Databricks)**

```sql
CREATE OR REPLACE TABLE users_dedup AS
SELECT id, email, name
FROM (
  SELECT 
    *,
    ROW_NUMBER() OVER (PARTITION BY email ORDER BY id) AS rn
  FROM users
)
WHERE rn = 1;

-- Drop old table, rename
DROP TABLE users;
ALTER TABLE users_dedup RENAME TO users;
```

---

#### **Solution 4: Keep Last Instead of First**

```sql
-- Keep the HIGHEST id (most recent)
CREATE OR REPLACE TABLE users_keep_latest AS
SELECT id, email, name
FROM (
  SELECT 
    *,
    ROW_NUMBER() OVER (PARTITION BY email ORDER BY id DESC) AS rn
  FROM users
)
WHERE rn = 1;
```

---

#### **Interview Follow-Up: Prevent Duplicates**

**Add UNIQUE constraint:**
```sql
ALTER TABLE users ADD CONSTRAINT unique_email UNIQUE (email);
```

**Use MERGE for upserts:**
```sql
MERGE INTO users AS target
USING new_users AS source
ON target.email = source.email
WHEN MATCHED THEN 
  UPDATE SET name = source.name
WHEN NOT MATCHED THEN
  INSERT (id, email, name) VALUES (source.id, source.email, source.name);
```

### ❓ Question 13: Calculate Cumulative Product

**Math/Analytics Interview Question:**
> "Given a table of daily growth rates, calculate the cumulative product (compound growth) over time. For example, day 1: 1.05, day 2: 1.03, cumulative = 1.05 * 1.03 = 1.0815."

### ✅ Answer 13: Cumulative Product (Logarithm Trick)

#### **Problem: SQL Has SUM, No PRODUCT**

SQL window functions don't have a `PRODUCT()` aggregate!

**Mathematical Identity:**
```
log(a × b × c) = log(a) + log(b) + log(c)

a × b × c = exp(log(a) + log(b) + log(c))
```

---

#### **Solution: LOG + SUM + EXP**

```sql
SELECT 
  date,
  growth_rate,
  EXP(SUM(LN(growth_rate)) OVER (ORDER BY date)) AS cumulative_growth
FROM daily_growth;
```

**Example:**
```
date       | growth_rate | cumulative_growth
2024-01-01 | 1.05        | 1.05
2024-01-02 | 1.03        | 1.0815  (1.05 × 1.03)
2024-01-03 | 1.02        | 1.1031  (1.05 × 1.03 × 1.02)
```

---

#### **Step-by-Step Breakdown**

```sql
SELECT 
  date,
  growth_rate,
  LN(growth_rate) AS log_rate,
  SUM(LN(growth_rate)) OVER (ORDER BY date) AS cumulative_log,
  EXP(SUM(LN(growth_rate)) OVER (ORDER BY date)) AS cumulative_product
FROM daily_growth;
```

**Output:**
```
date       | growth_rate | log_rate | cumulative_log | cumulative_product
2024-01-01 | 1.05        | 0.0488   | 0.0488         | 1.05
2024-01-02 | 1.03        | 0.0296   | 0.0784         | 1.0815
2024-01-03 | 1.02        | 0.0198   | 0.0982         | 1.1031
```

---

#### **Handle Negative Numbers (Use Power)**

**Problem:** `LN(negative)` is undefined!

```sql
SELECT 
  date,
  value,
  POWER(
    (SELECT PRODUCT(value) FROM table),  -- Not real syntax!
    1.0 / COUNT(*) OVER ()
  ) AS geometric_mean;
```

**Better:** Use PySpark UDF for arbitrary cumulative operations.

---

#### **Variation: Geometric Mean**

```sql
-- Geometric mean = (a × b × c)^(1/n)
SELECT 
  EXP(AVG(LN(value))) AS geometric_mean
FROM sales;
```

---

#### **Interview Follow-Up:**

**Q:** "What if growth_rate can be zero or negative?"

**A:** "LN() doesn't work for ≤ 0. I'd either:
1. Filter out invalid values
2. Use CASE to handle separately
3. Implement in PySpark with a custom UDF:

```python
from pyspark.sql import Window
from pyspark.sql.functions import col, expr
import numpy as np

# Custom cumulative product
window = Window.orderBy("date").rowsBetween(Window.unboundedPreceding, Window.currentRow)

df.withColumn(
  "cumulative_product",
  expr("aggregate(collect_list(growth_rate) OVER (ORDER BY date), 1.0, (acc, x) -> acc * x)")
)
```

### ❓ Question 14: Find All Possible Pairs (Combinations)

**Combinatorics Interview Question:**
> "Given a table of students, generate all possible 2-person study groups (pairs). Each pair should appear once (e.g., Alice-Bob, not also Bob-Alice)."

### ✅ Answer 14: All Pairs (Triangular Self-Join)

#### **Problem: Avoid Duplicate Pairs**

**Bad (duplicates):**
```
Alice-Bob
Bob-Alice  ← Duplicate!
Alice-Alice ← Invalid!
```

**Good:**
```
Alice-Bob
Alice-Carol
Bob-Carol
```

---

#### **Solution: Self-Join with <**

```sql
SELECT 
  s1.student_name AS student_1,
  s2.student_name AS student_2
FROM students s1
JOIN students s2 ON s1.student_id < s2.student_id  -- KEY!
ORDER BY s1.student_name, s2.student_name;
```

**Why `<` works:**
- `s1.id < s2.id` ensures each pair appears once
- Avoids self-pairs (Alice-Alice)
- Maintains order (always student1 < student2)

**Output:**
```
student_1 | student_2
Alice     | Bob
Alice     | Carol
Alice     | Dave
Bob       | Carol
Bob       | Dave
Carol     | Dave
```

---

#### **Variation: Generate All Triplets (3-Person Groups)**

```sql
SELECT 
  s1.student_name AS student_1,
  s2.student_name AS student_2,
  s3.student_name AS student_3
FROM students s1
JOIN students s2 ON s1.student_id < s2.student_id
JOIN students s3 ON s2.student_id < s3.student_id  -- Cascading <
ORDER BY s1.student_name, s2.student_name, s3.student_name;
```

**Output:**
```
student_1 | student_2 | student_3
Alice     | Bob       | Carol
Alice     | Bob       | Dave
Alice     | Carol     | Dave
Bob       | Carol     | Dave
```

---

#### **Count Total Combinations**

**Pairs:** C(n, 2) = n! / (2! × (n-2)!) = n × (n-1) / 2

```sql
-- Verify formula
WITH student_count AS (
  SELECT COUNT(*) AS n FROM students
)
SELECT 
  n * (n - 1) / 2 AS expected_pairs,
  (SELECT COUNT(*) FROM (
    SELECT 1 FROM students s1 JOIN students s2 ON s1.student_id < s2.student_id
  )) AS actual_pairs
FROM student_count;
```

---

#### **Interview Follow-Up: All Permutations (Order Matters)**

```sql
-- Generate Alice-Bob AND Bob-Alice
SELECT 
  s1.student_name AS person_1,
  s2.student_name AS person_2
FROM students s1
CROSS JOIN students s2
WHERE s1.student_id != s2.student_id;  -- Exclude self-pairs
```

**Permutations:** P(n, 2) = n × (n - 1)

### ❓ Question 15: Calculate Median Across Two Tables

**Advanced Analytics Question:**
> "You have two tables of sorted numbers. Find the median of the combined dataset WITHOUT merging the tables into one (optimize for large tables)."

### ✅ Answer 15: Median of Two Sorted Tables

#### **Naive Solution (Works but Slow)**

```sql
WITH combined AS (
  SELECT value FROM table1
  UNION ALL
  SELECT value FROM table2
),
ranked AS (
  SELECT 
    value,
    ROW_NUMBER() OVER (ORDER BY value) AS rn,
    COUNT(*) OVER () AS total
  FROM combined
)
SELECT AVG(value) AS median
FROM ranked
WHERE rn IN (FLOOR((total + 1) / 2.0), CEIL((total + 1) / 2.0));
```

**Problem:** UNION ALL forces full scan of both tables!

---

#### **Optimized Solution: Binary Search Approach**

**Key Insight:** Use counts without merging

```sql
WITH counts AS (
  SELECT COUNT(*) AS count1 FROM table1,
  SELECT COUNT(*) AS count2 FROM table2,
  SELECT count1 + count2 AS total FROM counts
),
median_position AS (
  SELECT 
    FLOOR((total + 1) / 2.0) AS pos_low,
    CEIL((total + 1) / 2.0) AS pos_high
  FROM counts
),
combined_with_rank AS (
  SELECT value, ROW_NUMBER() OVER (ORDER BY value) AS rn
  FROM (
    SELECT value FROM table1
    UNION ALL
    SELECT value FROM table2
  )
)
SELECT AVG(value) AS median
FROM combined_with_rank
WHERE rn BETWEEN (
  SELECT pos_low FROM median_position
) AND (
  SELECT pos_high FROM median_position
);
```

---

#### **For HUGE Tables: Sample + Approximate**

```sql
-- Use APPROX_PERCENTILE for massive datasets
WITH combined_sample AS (
  SELECT value FROM table1 TABLESAMPLE (10 PERCENT)
  UNION ALL
  SELECT value FROM table2 TABLESAMPLE (10 PERCENT)
)
SELECT APPROX_PERCENTILE(value, 0.5) AS approximate_median
FROM combined_sample;
```

**Trade-off:** 90% faster, ~5% error margin

---

#### **Interview Discussion Points:**

1. **Why not UNION?** "Full table scan is expensive; if tables are pre-sorted, we could binary search."
2. **Distributed systems?** "In Spark, union shuffles data; better to use approximate percentiles."
3. **Exact vs Approximate?** "For billions of rows, APPROX_PERCENTILE with 1% sample is acceptable."

## 🎓 Congratulations - You've Mastered Complex SQL!

### 📊 What You've Learned:

✅ **Recursive CTEs** - Hierarchies, graphs, BOM explosion
✅ **Advanced Window Functions** - Gaps/islands, running totals, conditional resets
✅ **Self-Joins** - Overlaps, sequences, pairs
✅ **Pivoting/Unpivoting** - Dynamic data reshaping
✅ **Classic Brain Teasers** - Nth highest, deduplication, cumulative product

---

### 🚀 Next Steps:

1. **Practice on LeetCode** - SQL 50, Database category
2. **Build Real Projects** - Apply these patterns to actual data
3. **Study Query Plans** - Understand EXPLAIN output
4. **Learn Spark SQL** - Master distributed query optimization
5. **Read Documentation** - Databricks, Spark, Delta Lake docs

---

### 🎯 Interview Day Checklist:

✅ Can you explain recursive CTEs without code?
✅ Do you know when to use RANK vs DENSE_RANK vs ROW_NUMBER?
✅ Can you solve "gaps and islands" from memory?
✅ Can you write a self-join for overlapping intervals?
✅ Do you understand the log-trick for cumulative products?

---

**You're now ready for senior+ data engineering SQL interviews!** 🎉

Good luck! 🚀